# 03 — Kernel methods, random forest and XGBoost

Fits the **additive kernel** (readable), **kernel logistic regression**, **Nadaraya–Watson**, **random forest** and **XGBoost** (flexible references) on every feature set, with the same folds and descriptor lists as notebook 02, and saves outputs in the shared format. The kernel formulations are written out before their code. All analysis and plots are in notebook 04.

**Using Sean's Python implementations instead:** any model can be fitted elsewhere as long as it writes `outputs/cv_<model>.csv`, `oof_<model>.csv` and `test_<model>.csv` in the format of the README, using `outputs/folds.csv` and `outputs/feature_sets.csv`. Then remove it from `RUN_MODELS` here.

In [1]:
include("common.jl")
C = setup()

RUN_MODELS = ["Additive kernel", "Kernel logistic", "Nadaraya-Watson", "Random forest", "XGBoost"]
RUN_SETS   = SETS
KLR_GRID = vec([(g = g, lambda = l) for g in [0.3, 1.0, 3.0], l in [0.01, 0.1, 1.0]])
NW_GRID  = [(g = g,) for g in [1.0, 3.0, 10.0, 30.0]]
ADD_GRID = vec([(gamma = γ, lambda = l) for γ in [0.1, 0.3, 1.0], l in [0.1, 1.0, 10.0]])
RF_TREES,  RF_GRID  = 500, vec([(max_depth = d, minbucket = m) for d in [6, 10], m in [1, 5]])
XGB_ROUNDS, XGB_GRID = 300, vec([(max_depth = d, eta = e) for d in [2, 4, 6], e in [0.05, 0.1]])

common.jl loaded: 7 feature sets, outputs in 'outputs', figures in 'figures'.


(300, @NamedTuple{max_depth::Int64, eta::Float64}[(max_depth = 2, eta = 0.05), (max_depth = 4, eta = 0.05), (max_depth = 6, eta = 0.05), (max_depth = 2, eta = 0.1), (max_depth = 4, eta = 0.1), (max_depth = 6, eta = 0.1)])

---
## Kernel methods — formulations

All three measure how **similar** two chemicals are, after standardizing each descriptor on the training part (zero mean, unit spread) so no descriptor dominates because of its units.

**1. Gaussian kernel.** For standardized descriptor vectors $z, z'$ over the $p$ descriptors in the list,

$$K(z, z') = \exp\!\big(-\gamma\, \lVert z - z' \rVert^2\big), \qquad \gamma = g / p .$$

$K = 1$ for identical chemicals and falls toward 0 as they differ; large $\gamma$ means only very close chemicals count. Dividing by $p$ keeps $g$ meaningful at every budget, since $\lVert z - z'\rVert^2$ grows roughly in proportion to $p$.

**2. Kernel logistic regression (flexible).** Instead of a weighted sum of descriptors, a weighted sum of **similarities to the training chemicals**:

$$P(\text{NRB}\mid z) = \sigma\big(f(z)\big), \qquad f(z) = b + \sum_{i=1}^{n} \alpha_i\, K(z, z_i), \qquad \sigma(t) = \frac{1}{1+e^{-t}},$$

$$\min_{\alpha,\,b}\; \sum_{i=1}^{n}\Big[\log\big(1+e^{\eta_i}\big) - y_i\,\eta_i\Big] + \frac{\lambda}{2}\,\alpha^\top K\alpha, \qquad \eta = K\alpha + b\mathbf{1}.$$

By the representer theorem this is the best smooth function in the space the kernel defines, so the boundary can be **curved** and depend on **combinations** of descriptors; $\lambda$ controls smoothness. *Solved by Newton's method:* with $p_i = \sigma(\eta_i)$, $w_i = p_i(1-p_i)$, $W = \operatorname{diag}(w)$, each step solves

$$\begin{bmatrix} WK + \lambda I & w \\ w^\top K & \sum_i w_i \end{bmatrix}\begin{bmatrix}\alpha^{\text{new}}\\ b^{\text{new}}\end{bmatrix} = \begin{bmatrix} W\eta + (y-p) \\ \sum_i\big[w_i\eta_i + (y_i - p_i)\big]\end{bmatrix}.$$

*Not readable:* a prediction depends on all ~675 training chemicals at once.

**3. Nadaraya–Watson (flexible, simplest).** A similarity-weighted average of the training labels, with only $\gamma$ tuned:

$$\hat P(\text{NRB}\mid z) = \frac{\sum_i K(z, z_i)\, y_i}{\sum_i K(z, z_i)} .$$

**4. Additive kernel (readable).** One kernel per descriptor, added up:

$$K_{\text{add}}(z, z') = \sum_{j=1}^{p} \exp\!\big(-\gamma (z_j - z'_j)^2\big), \qquad f(z) = b + \sum_{j=1}^{p} f_j(z_j), \quad f_j(z_j) = \sum_i \alpha_i\, e^{-\gamma (z_j - z_{ij})^2}.$$

Fitted exactly like kernel logistic regression with $K_{\text{add}}$. The score is a **sum of one smooth curve per descriptor**, so each $f_j$ can be plotted (notebook 04); it cannot capture interactions, which is the price of readability. It uses the shared descriptor lists (Sean's earlier version also chose its own descriptors with binaries $s_j$, $\sum_j s_j \le k$).

**Tuning (mean 5-fold AUC):** kernel logistic $g\in\{0.3,1,3\}$, $\lambda\in\{0.01,0.1,1\}$; Nadaraya–Watson $g\in\{1,3,10,30\}$; additive kernel $\gamma\in\{0.1,0.3,1\}$, $\lambda\in\{0.1,1,10\}$.

## Kernel methods — implementation

In [2]:
sqdist(A, B) = max.(sum(A .^ 2, dims = 2) .+ sum(B .^ 2, dims = 2)' .- 2 .* (A * B'), 0.0)
gauss_kernel(A, B, γ)    = exp.(-γ .* sqdist(A, B))
additive_kernel(A, B, γ) = sum(exp.(-γ .* (A[:, j] .- B[:, j]') .^ 2) for j in 1:size(A, 2))

function klr_fit(K, y, λ; iters = 100)              # Newton step of formulation 2
    n = length(y); a = zeros(n); b = 0.0; yf = Float64.(y)
    A = zeros(n + 1, n + 1)
    for _ in 1:iters
        η = K * a .+ b
        p = 1 ./ (1 .+ exp.(-η))
        w = clamp.(p .* (1 .- p), 1e-10, Inf)
        rhs = w .* η .+ (yf .- p)
        A[1:n, 1:n] .= w .* K
        for i in 1:n
            A[i, i] += λ
        end
        A[1:n, n + 1] .= w
        A[n + 1, 1:n] .= vec(w' * K)
        A[n + 1, n + 1] = sum(w)
        sol = A \ vcat(rhs, sum(rhs))
        step = maximum(abs.(sol[1:n] .- a))
        a, b = sol[1:n], sol[n + 1]
        step < 1e-8 && break
    end
    a, b
end

function fit_kernel(X, y, fs, prm; kind)
    mu, sd = standardize_fit(X, fs)
    Z = zs(X, fs, mu, sd)
    kern = kind == "Additive kernel" ? additive_kernel : gauss_kernel
    γ = kind == "Additive kernel" ? prm.gamma : prm.g / length(fs)
    a, b = kind == "Nadaraya-Watson" ? (Float64[], 0.0) : klr_fit(kern(Z, Z, γ), y, prm.lambda)
    (kind = kind, fs = fs, mu = mu, sd = sd, Z = Z, y = Float64.(y), γ = γ, kern = kern, a = a, b = b)
end
function predict_kernel(m, X)
    Kv = m.kern(zs(X, m.fs, m.mu, m.sd), m.Z, m.γ)
    m.kind == "Nadaraya-Watson" ? (Kv * m.y) ./ max.(vec(sum(Kv, dims = 2)), 1e-300) : 1 ./ (1 .+ exp.(-(Kv * m.a .+ m.b)))
end

predict_kernel (generic function with 1 method)

---
## Random forest and XGBoost

Ensembles of many trees (random forest: trees on random resamples, averaged; XGBoost: trees added one after another, each correcting the last), fitted with IAI's wrappers and tuned by mean 5-fold AUC. If IAI rejects a parameter name, check `?IAI.RandomForestClassifier` / `?IAI.XGBoostClassifier`.

In [3]:
function fit_rf(X, y, fs, prm)
    lnr = IAI.RandomForestClassifier(; random_seed = SEED, num_trees = RF_TREES, prm...)
    IAI.fit!(lnr, X[:, fs], to_labels(y)); (lnr = lnr, fs = fs)
end
function fit_xgb(X, y, fs, prm)
    lnr = IAI.XGBoostClassifier(; random_seed = SEED, num_round = XGB_ROUNDS, prm...)
    IAI.fit!(lnr, X[:, fs], to_labels(y)); (lnr = lnr, fs = fs)
end
predict_iai(m, X) = IAI.predict_proba(m.lnr, X[:, m.fs])[!, "NRB"]

kfit(kind) = (X, y, fs, prm) -> fit_kernel(X, y, fs, prm; kind = kind)
SPEC = Dict("Additive kernel" => (ADD_GRID, kfit("Additive kernel"), predict_kernel),
            "Kernel logistic" => (KLR_GRID, kfit("Kernel logistic"), predict_kernel),
            "Nadaraya-Watson" => (NW_GRID,  kfit("Nadaraya-Watson"), predict_kernel),
            "Random forest"   => (RF_GRID,  fit_rf,  predict_iai),
            "XGBoost"         => (XGB_GRID, fit_xgb, predict_iai))

Dict{String, Tuple{Vector, Function, Function}} with 5 entries:
  "Additive kernel" => (@NamedTuple{gamma::Float64, lambda::Float64}[(gamma = 0…
  "Kernel logistic" => (@NamedTuple{g::Float64, lambda::Float64}[(g = 0.3, lamb…
  "Nadaraya-Watson" => (@NamedTuple{g::Float64}[(g = 1.0,), (g = 3.0,), (g = 10…
  "XGBoost"         => (@NamedTuple{max_depth::Int64, eta::Float64}[(max_depth …
  "Random forest"   => (@NamedTuple{max_depth::Int64, minbucket::Int64}[(max_de…

---
## Run and save

In [4]:
final_kernel = Dict{Tuple{String,String},Any}()
for m in RUN_MODELS
    st = new_store()
    grid, fitfun, predfun = SPEC[m]
    for s in RUN_SETS
        r = cv_generic(C, s, grid, fitfun, predfun)
        record!(st, C, m, s, r)
        final_kernel[(s, m)] = r.model
        println(rpad(m, 16), rpad(setlabel(s), 12), "CV AUC ", round(r.cv_auc, digits = 3), " ± ", round(r.cv_sd, digits = 3),
                " | ", r.settings, " | ", round(r.time, digits = 1), " s")
    end
    save_store(st, m)
end

Additive kernel k=3 reg.    CV AUC 0.87 ± 0.024 | gamma=1.0, lambda=0.1 | 10.5 s
Additive kernel k=3 log.    CV AUC 0.874 ± 0.019 | gamma=0.1, lambda=0.1 | 3.7 s
Additive kernel k=5 reg.    CV AUC 0.887 ± 0.016 | gamma=0.1, lambda=1.0 | 4.4 s
Additive kernel k=5 log.    CV AUC 0.899 ± 0.02 | gamma=0.1, lambda=0.1 | 4.7 s
Additive kernel k=10 reg.   CV AUC 0.913 ± 0.021 | gamma=0.1, lambda=0.1 | 7.1 s
Additive kernel k=10 log.   CV AUC 0.92 ± 0.015 | gamma=1.0, lambda=0.1 | 7.1 s
Additive kernel all 41      CV AUC 0.936 ± 0.012 | gamma=1.0, lambda=1.0 | 17.3 s
Saved outputs for Additive kernel → outputs/*_additive_kernel.csv
Kernel logistic k=3 reg.    CV AUC 0.875 ± 0.019 | g=1.0, lambda=1.0 | 3.9 s
Kernel logistic k=3 log.    CV AUC 0.893 ± 0.016 | g=1.0, lambda=0.01 | 3.9 s
Kernel logistic k=5 reg.    CV AUC 0.899 ± 0.025 | g=3.0, lambda=0.1 | 3.5 s
Kernel logistic k=5 log.    CV AUC 0.908 ± 0.026 | g=3.0, lambda=0.1 | 3.7 s
Kernel logistic k=10 reg.   CV AUC 0.926 ± 0.018 | g=3.0, l

[ Warning: This copy of Interpretable AI software is for academic purposes only and not for commercial use.


Random forest   k=3 reg.    CV AUC 0.878 ± 0.016 | max_depth=6, minbucket=1 | 18.1 s
Random forest   k=3 log.    CV AUC 0.894 ± 0.016 | max_depth=6, minbucket=1 | 11.0 s
Random forest   k=5 reg.    CV AUC 0.905 ± 0.017 | max_depth=10, minbucket=1 | 12.5 s
Random forest   k=5 log.    CV AUC 0.911 ± 0.015 | max_depth=10, minbucket=5 | 13.1 s
Random forest   k=10 reg.   CV AUC 0.922 ± 0.011 | max_depth=10, minbucket=1 | 15.2 s
Random forest   k=10 log.   CV AUC 0.926 ± 0.012 | max_depth=10, minbucket=1 | 14.8 s
Random forest   all 41      CV AUC 0.94 ± 0.01 | max_depth=10, minbucket=1 | 20.2 s
Saved outputs for Random forest → outputs/*_random_forest.csv
XGBoost         k=3 reg.    CV AUC 0.88 ± 0.012 | max_depth=2, eta=0.05 | 15.2 s
XGBoost         k=3 log.    CV AUC 0.89 ± 0.019 | max_depth=4, eta=0.05 | 12.3 s
XGBoost         k=5 reg.    CV AUC 0.9 ± 0.02 | max_depth=4, eta=0.05 | 14.0 s
XGBoost         k=5 log.    CV AUC 0.9 ± 0.019 | max_depth=4, eta=0.05 | 14.4 s
XGBoost         k=1

---
## Additive-kernel curves (saved for notebook 04)

For the final additive models at k = 3 (both selectors): the curve $f_j$ of each descriptor over its observed range, centered on its average over the development set. Saved to `outputs/additive_curves.csv`; plotted in notebook 04.

In [5]:
curves = DataFrame(set = String[], descriptor = String[], x = Float64[], effect = Float64[])
for s in ["reg_k3", "log_k3"]
    haskey(final_kernel, (s, "Additive kernel")) || continue
    am = final_kernel[(s, "Additive kernel")]
    for (j, c) in enumerate(am.fs)
        xr = Float64.(C.X_dev[!, c])
        grid = collect(range(minimum(xr), maximum(xr), length = 300))
        f(zv) = exp.(-am.γ .* (zv .- am.Z[:, j]') .^ 2) * am.a
        eff = f((grid .- am.mu[j]) ./ am.sd[j]) .- mean(f(am.Z[:, j]))
        append!(curves, DataFrame(set = fill(s, 300), descriptor = fill(c, 300), x = grid, effect = eff))
    end
end
CSV.write(joinpath(OUT_DIR, "additive_curves.csv"), curves)
println("Saved additive-kernel curves for: ", join(unique(curves.set), ", "))

Saved additive-kernel curves for: reg_k3, log_k3
